In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict
import re

def plot_clip_rewards(json_paths, figsize=(16, 8), save_path=None, labels=None):
    """
    Plot CLIP rewards grouped by prompts across multiple experiments.
    Prompts are sorted by the first experiment's rewards for better comparison.
    
    Parameters:
    -----------
    json_paths : list of str
        List of paths to clip_rewards.json files
    figsize : tuple
        Figure size (width, height)
    save_path : str, optional
        Path to save the figure
    """
    # Prepare data structure
    all_data = {}
    
    # Read all JSON files
    for json_path in json_paths:
        # Extract experiment name
        path = Path(json_path)
        exp_name = path.parent.parent.name
        
        # Read JSON
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        # Group scores by prompt and calculate mean
        prompt_scores = defaultdict(list)
        for prompt, score in zip(data['prompts'], data['all_scores']):
            prompt_scores[prompt].append(score)
        
        # Calculate mean for each prompt
        prompt_means = {prompt: np.mean(scores) 
                       for prompt, scores in prompt_scores.items()}
        
        all_data[exp_name] = prompt_means
    
    # Get all unique prompts
    all_prompts_set = set(prompt for exp_data in all_data.values() 
                         for prompt in exp_data.keys())
    
    # Sort prompts by the first experiment's rewards (descending order)
    first_exp_name = list(all_data.keys())[0]
    first_exp_data = all_data[first_exp_name]
    
    # Sort: prompts in first experiment by their reward, then prompts not in first experiment alphabetically
    prompts_in_first = sorted(
        [p for p in all_prompts_set if p in first_exp_data],
        key=lambda p: first_exp_data[p],
        reverse=True
    )
    prompts_not_in_first = sorted([p for p in all_prompts_set if p not in first_exp_data])
    all_prompts = (prompts_in_first + prompts_not_in_first)[-10:]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot each experiment
    colors = plt.cm.tab10(np.linspace(0, 1, len(all_data)))
    x_positions = np.arange(len(all_prompts))
    
    for idx, (exp_name, prompt_means) in enumerate(all_data.items()):
        # Get means for all prompts (0 if prompt not in this experiment)
        y_values = [prompt_means.get(prompt, np.nan) for prompt in all_prompts]
        display_name = labels.get(exp_name, exp_name) if labels else exp_name

        
        ax.plot(x_positions, y_values, marker='o', linewidth=2, 
                markersize=6, label=display_name, color=colors[idx], alpha=0.8)
    
    # Customize x-axis
    # Truncate long prompts for display
    display_labels = [prompt[:30] + '...' if len(prompt) > 30 else prompt 
                     for prompt in all_prompts]
    
    ax.set_xticks(x_positions)
    ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=9)
    
    # Labels and title
    ax.set_xlabel('Prompts (sorted by first experiment)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean CLIP Reward', fontsize=12, fontweight='bold')
    ax.set_title('CLIP Rewards by Prompt Across Experiments', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # Grid and legend
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='best', framealpha=0.9, fontsize=10)
    
    # Tight layout
    plt.tight_layout()
    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    plt.show()
    
    # Print summary statistics
    print("\n=== Summary Statistics ===")
    for exp_name, prompt_means in all_data.items():
        values = list(prompt_means.values())
        print(f"\n{exp_name}:")
        print(f"  Number of prompts: {len(values)}")
        print(f"  Mean reward: {np.mean(values):.4f}")
        print(f"  Std reward: {np.std(values):.4f}")
        print(f"  Min reward: {np.min(values):.4f}")
        print(f"  Max reward: {np.max(values):.4f}")

def plot_rarity_score(json_paths, figsize=(12, 6), save_path=None, labels=None):
    """
    Plot a rarity score for each experiment as a styled bar chart (in %).
    
    Rarity score = mean(experiment_score - pretrained_score) × 100
                   over the last 10 prompts (lowest-scoring from first experiment).
    'pretrained' always gets score 0.
    """
    # ── 1. Load all data ──────────────────────────────────────────────────────
    all_data = {}
    for json_path in json_paths:
        path = Path(json_path)
        exp_name = path.parent.parent.name
        with open(json_path, 'r') as f:
            data = json.load(f)

        prompt_scores = defaultdict(list)
        for prompt, score in zip(data['prompts'], data['all_scores']):
            prompt_scores[prompt].append(score)

        prompt_means = {prompt: np.mean(scores)
                        for prompt, scores in prompt_scores.items()}
        all_data[exp_name] = prompt_means

    # ── 2. Identify the last 10 prompts — always from pretrained ─────────────
    if 'pretrained' not in all_data:
        raise ValueError("No experiment named 'pretrained' found. "
                         "Ensure one folder is named 'pretrained'.")

    pretrained_scores = all_data['pretrained']

    # Sort pretrained's prompts by score (desc), with prompt text as tiebreaker
    prompts_sorted = sorted(
        pretrained_scores.keys(),
        key=lambda p: (pretrained_scores[p], p),
        reverse=True
    )
    last_10_prompts = prompts_sorted[-10:]   # 10 lowest-scoring prompts in pretrained

    # ── 3. Check pretrained exists ────────────────────────────────────────────
    if 'pretrained' not in all_data:
        raise ValueError("No experiment named 'pretrained' found. "
                         "Ensure one folder is named 'pretrained'.")

    pretrained_scores = all_data['pretrained']

    # ── 4. Compute rarity scores (in %) ──────────────────────────────────────
    exp_names     = list(all_data.keys())
    rarity_scores = []
    for exp_name in exp_names:
        exp_scores = all_data[exp_name]
        diffs = [exp_scores.get(p, np.nan) - pretrained_scores.get(p, np.nan)
                 for p in last_10_prompts]
        rarity_scores.append(np.nanmean(diffs) * 100)

    sorted_pairs = sorted(zip(exp_names, rarity_scores), key=lambda x: x[1], reverse=True)
    exp_names, rarity_scores = zip(*sorted_pairs)
    exp_names, rarity_scores = list(exp_names), list(rarity_scores)

    # ── 5. Plot ───────────────────────────────────────────────────────────────
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#f8f9fa')

    # Color palette (light-friendly)
    palette = {
        'pretrained': '#adb5bd',
        'positive':   '#2a9d8f',
        'negative':   '#e76f51',
    }
    bar_colors = [
        palette['pretrained'] if name == 'pretrained' else
        (palette['positive'] if score >= 0 else palette['negative'])
        for name, score in zip(exp_names, rarity_scores)
    ]

    x = np.arange(len(exp_names))
    bars = ax.bar(x, rarity_scores, color=bar_colors, edgecolor='white',
                  linewidth=1.2, width=0.55, zorder=3)

    # ── Value labels — always inside the plot, anchored to bar top/bottom ────
    # ── Value labels — outside the bars ──────────────────────────────────────
    y_min, y_max = min(rarity_scores), max(rarity_scores)
    y_pad = (y_max - y_min) * 0.08 if y_max != y_min else 1.0
    for bar, score in zip(bars, rarity_scores):
        bar_h  = bar.get_height()
        bar_cx = bar.get_x() + bar.get_width() / 2
        if score >= 0:
            label_y = bar_h + y_pad * 0.3   # just above the top of the bar
            va = 'bottom'
        else:
            label_y = bar_h - y_pad * 0.3   # just below the bottom of the bar
            va = 'top'
        ax.text(bar_cx, label_y,
                f'{score:+.2f}%',
                ha='center', va=va,
                fontsize=9, fontweight='bold',
                color='#333333')
    # extend y limits so labels are never cut off
    ax.set_ylim(y_min - y_pad * 3,
                y_max + y_pad * 3)

    # ── Axis styling ──────────────────────────────────────────────────────────
    ax.axhline(0, color='#495057', linewidth=1.2, linestyle='--', zorder=4)

    ax.set_xticks(x)
    display_names = [labels.get(n, n) if labels else n for n in exp_names]
    ax.set_xticklabels(display_names, rotation=30, ha='right', fontsize=9,
                       color='#333333')
    ax.tick_params(axis='y', colors='#333333', labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#cccccc')
    ax.grid(True, axis='y', color='#dee2e6', linewidth=0.8, zorder=0)

    ax.set_ylim(min(rarity_scores) - y_pad * 2,
                max(rarity_scores) + y_pad * 2)

    ax.set_ylabel('Rarity Score (%)\nvs pretrained', fontsize=11,
                  fontweight='bold', color='#333333', labelpad=10)
    ax.set_title('Rarity Score per Experiment\n'
                 'mean CLIP reward Δ over rare prompts (last 10), relative to pretrained',
                 fontsize=13, fontweight='bold', color='#1a1a2e', pad=16)

    # ── Legend ────────────────────────────────────────────────────────────────
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=palette['positive'],   label='Better than pretrained'),
        Patch(facecolor=palette['negative'],   label='Worse than pretrained'),
        Patch(facecolor=palette['pretrained'], label='Pretrained (baseline = 0)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', framealpha=0.9,
              fontsize=9, facecolor='white', edgecolor='#cccccc')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        print(f"Rarity score plot saved to {save_path}")

    plt.show()

    # ── Summary ───────────────────────────────────────────────────────────────
    print("\n=== Rarity Scores (%) — mean CLIP Δ vs pretrained, last 10 prompts ===")
    for name, score in zip(exp_names, rarity_scores):
        marker = '▲' if score > 0 else ('▼' if score < 0 else '●')
        print(f"  {marker} {name:40s}: {score:+.2f}%")

def plot_clip_rewards_heatmap(json_paths, figsize=(14, 8), save_path=None):
    """
    Plot CLIP rewards as a heatmap for better visualization with many prompts.
    Prompts are sorted by the first experiment's rewards.
    """
    # Prepare data
    all_data = {}
    
    for json_path in json_paths:
        path = Path(json_path)
        exp_name = path.parent.parent.name
        
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        prompt_scores = defaultdict(list)
        for prompt, score in zip(data['prompts'], data['all_scores']):
            prompt_scores[prompt].append(score)
        
        prompt_means = {prompt: np.mean(scores) 
                       for prompt, scores in prompt_scores.items()}
        
        all_data[exp_name] = prompt_means
    
    # Get all unique prompts and sort by first experiment
    all_prompts_set = set(prompt for exp_data in all_data.values() 
                         for prompt in exp_data.keys())
    
    first_exp_name = list(all_data.keys())[0]
    first_exp_data = all_data[first_exp_name]
    
    prompts_in_first = sorted(
        [p for p in all_prompts_set if p in first_exp_data],
        key=lambda p: first_exp_data[p],
        reverse=True
    )
    prompts_not_in_first = sorted([p for p in all_prompts_set if p not in first_exp_data])
    all_prompts = prompts_in_first + prompts_not_in_first
    
    # Create matrix
    exp_names = list(all_data.keys())
    matrix = np.zeros((len(exp_names), len(all_prompts)))
    
    for i, exp_name in enumerate(exp_names):
        for j, prompt in enumerate(all_prompts):
            matrix[i, j] = all_data[exp_name].get(prompt, np.nan)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=figsize)
    
    im = ax.imshow(matrix, aspect='auto', cmap='viridis', interpolation='nearest')
    
    # Set ticks
    ax.set_xticks(np.arange(len(all_prompts)))
    ax.set_yticks(np.arange(len(exp_names)))
    
    # Labels
    display_labels = [prompt[:40] + '...' if len(prompt) > 40 else prompt 
                     for prompt in all_prompts]
    ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(exp_names, fontsize=10)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Mean CLIP Reward', rotation=270, labelpad=20, fontsize=11)
    
    # Title
    ax.set_title('CLIP Rewards Heatmap: Experiments vs Prompts (sorted by first exp)', 
                 fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Heatmap saved to {save_path}")
    
    plt.show()


def plot_inception_scores(inception_scores_dict, figsize=(10, 6), save_path=None, labels=None):
    """
    Plot inception scores across experiments as a bar chart.
    
    Parameters:
    -----------
    inception_scores_dict : dict
        Dictionary mapping experiment names to inception scores
        e.g., {'b2diffu_rl': 1.2858, 'only_5_steps': 1.2845, ...}
    figsize : tuple
        Figure size (width, height)
    save_path : str, optional
        Path to save the figure
    """
    # Sort experiments by inception score (descending)
    sorted_items = sorted(inception_scores_dict.items(), key=lambda x: x[1], reverse=True)
    exp_names = [item[0] for item in sorted_items]
    scores = [item[1] for item in sorted_items]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Create bar chart
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(exp_names)))
    bars = ax.bar(range(len(exp_names)), scores, color=colors, alpha=0.8, edgecolor='black')
    
    # Add value labels on bars
    for i, (bar, score) in enumerate(zip(bars, scores)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{score:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Customize plot
    ax.set_xticks(range(len(exp_names)))
    display_names = [labels.get(n, n) if labels else n for n in exp_names]

    ax.set_xticklabels(display_names, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Inception Score', fontsize=12, fontweight='bold')
    ax.set_title('Inception Scores Across Experiments', fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--', axis='y')
    
    # Set y-axis to start slightly below minimum for better visualization
    y_min = min(scores) * 0.99
    y_max = max(scores) * 1.01
    ax.set_ylim(y_min, y_max)
    
    plt.tight_layout()
    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Inception scores plot saved to {save_path}")
    
    plt.show()
    
    # Print statistics
    print("\n=== Inception Score Statistics ===")
    print(f"Best: {exp_names[0]} = {scores[0]:.4f}")
    print(f"Worst: {exp_names[-1]} = {scores[-1]:.4f}")
    print(f"Mean: {np.mean(scores):.4f}")
    print(f"Std: {np.std(scores):.4f}")
    print(f"Range: {scores[0] - scores[-1]:.4f}")


def plot_mean_rewards(mean_rewards_dict, figsize=(10, 6), save_path=None, labels=None):
    """
    Plot inception scores across experiments as a bar chart.
    
    Parameters:
    -----------
    mean_rewards_dict : dict
        Dictionary mapping experiment names to inception scores
        e.g., {'b2diffu_rl': 1.2858, 'only_5_steps': 1.2845, ...}
    figsize : tuple
        Figure size (width, height)
    save_path : str, optional
        Path to save the figure
    """
    # Sort experiments by inception score (descending)
    sorted_items = sorted(mean_rewards_dict.items(), key=lambda x: x[1], reverse=True)
    exp_names = [item[0] for item in sorted_items]
    scores = [item[1] for item in sorted_items]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Create bar chart
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(exp_names)))
    bars = ax.bar(range(len(exp_names)), scores, color=colors, alpha=0.8, edgecolor='black')
    
    # Add value labels on bars
    for i, (bar, score) in enumerate(zip(bars, scores)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{score:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Customize plot
    ax.set_xticks(range(len(exp_names)))
    display_names = [labels.get(n, n) if labels else n for n in exp_names]

    ax.set_xticklabels(display_names, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Mean Rewards', fontsize=12, fontweight='bold')
    ax.set_title('Mean Rewards Across Experiments', fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--', axis='y')
    
    # Set y-axis to start slightly below minimum for better visualization
    y_min = min(scores) * 0.99
    y_max = max(scores) * 1.01
    ax.set_ylim(y_min, y_max)
    
    plt.tight_layout()
    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Mean Rewards plot saved to {save_path}")
    
    plt.show()
    
    # Print statistics
    print("\n=== Mean Reward Statistics ===")
    print(f"Best: {exp_names[0]} = {scores[0]:.4f}")
    print(f"Worst: {exp_names[-1]} = {scores[-1]:.4f}")
    print(f"Mean: {np.mean(scores):.4f}")
    print(f"Std: {np.std(scores):.4f}")
    print(f"Range: {scores[0] - scores[-1]:.4f}")


# Example usage:
if __name__ == "__main__":
    def extract_mean_from_file(file_path):
        with open(file_path, 'r') as file:
            content = file.read()
        
        # \s* matches any whitespace, ([\d.]+) captures the digits and decimal point
        match = re.search(r'Mean:\s*([\d.]+)', content)
        
        if match:
            return float(match.group(1))
        return None
    
    def extract_mean_reward_from_file(clip_file_path):
        import json
        with open(clip_file_path, "r") as f:
            data = json.load(f)
        return data["clip_reward_mean"]
        

    # Example paths
    # json_paths = [
    #     "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/b2diffu_rl/clip_rewards.json",
    #     "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/only_5_steps/clip_rewards.json",
    #     "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/another_only_5_steps/clip_rewards.json",
    #     "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/no_branch_yes_selection_only_5_steps/clip_rewards.json",
    #     "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/fk_steering/clip_rewards.json",
    # ]
    # run_names = ["pretrained", "vanilla_ddpo", "another_only_5_steps", "5_only_lambda_2_fk_4particles", "10_only_lambda_2_fk_4particles", "best_worst_lambda_10_fk_4particles", "lambda_2_fk_4particles", "lambda_10_fk_4particles"]
    # stage_number = [0, 21, 30, 45, 30, 10, 16, 13]
    # run_names = ["vanilla_ddpo", "lambda_2_fk_4particles", "lambda_10_fk_4particles"]
    # stage_number = [21, 16, 13]
    # run_names = ["pretrained","vanilla_ddpo", "b2diffu_try2", "10_only_lambda_2_fk_4particles", "lambda_2_fk_4particles"]
    # stage_number = [0, 21, 23, 30, 23]

    run_names = ["pretrained","vanilla_ddpo", "b2diffu_try2", "10_only_lambda_2_fk_4particles", "lambda_2_fk_4particles", "15_only_lambda_2_fk_4particles", "20_only_lambda_2_fk_4particles", "incremental_5_10_15_only_lambda_2_fk_4particles", "best_worst_lambda_2_fk_4particles"]
    stage_number = [0, 21, 23, 30, 23, 28, 26, 35, 13]
    # run_names = ["pretrained","vanilla_ddpo", "b2diffu_try2", "10_only_lambda_2_fk_4particles", "15_only_lambda_2_fk_4particles", "20_only_lambda_2_fk_4particles"]
    # stage_number = [0, 21, 23, 30, 28, 26]

    run_names = ["pretrained","vanilla_ddpo", "b2diffu_try2", "10_only_lambda_2_fk_4particles", "lambda_2_fk_4particles", "incremental_5_10_15_only_lambda_2_fk_4particles", "best_worst_lambda_2_fk_4particles"]
    stage_number = [0, 21, 23, 30, 23, 35, 13]

    run_names = ["pretrained", "b2diffu_try2","custom_bpt_incremental_4_8_12_16_fk","fk_then_branch_run","new_incremental_4_8_12_16_only_lambda_2_fk_4particles"]
    stage_number = [0, 23, 11, 21, 35]

    run_names = ["pretrained", "b2diffu_try2","new_incremental_4_8_12_16_only_lambda_2_fk_4particles", "custom_bpt_incremental_4_8_12_16_fk"]
    stage_number = [0, 23, 35, 13]

    run_names = ["pretrained", "b2diffu_try2", "incremental_5_10_15_only_lambda_2_fk_4particles", "new_incremental_4_8_12_16_only_lambda_2_fk_4particles"]
    stage_number = [0, 23, 41, 35]


    # Feb 20

    run_names = ["pretrained", "b2diffu_try2", "5_only_lambda_2_fk_4particle_seed_69", "always_split_15_b2_only_5","always_split_15_b2_inc", "all_norm_inc", "only_10_branch_lambda_2_fk_4particles"]
    stage_number = [0, 23, 33, 37, 34, 33, 32]

    run_names = ["pretrained", "b2diffu_try2", "always_split_15_b2_only_5","always_split_15_b2_inc", "only_10_branch_lambda_2_fk_4particles", "incremental_branch_lambda_2_fk_4particles"]
    stage_number = [0, 23, 29, 29, 31, 32]

    run_names = ["pretrained", "b2diffu_try2", "only_10_branch_lambda_2_fk_4particles", "incremental_branch_lambda_2_fk_4particles"]
    stage_number = [0, 23, 31, 32]

    run_names = ["pretrained", "b2diffu_try2", "only_10_branch_lambda_2_fk_4particles", "incremental_branch_lambda_2_fk_4particles"]
    stage_number = [0, 23, 31, 32]

    run_names = ["pretrained", "b2diffu_try2", "always_split_15_b2_only_5","always_split_15_b2_inc", "only_10_branch_lambda_2_fk_4particles", "incremental_branch_lambda_2_fk_4particles", "5_only_lambda_2_fk_4particle_seed_69"]
    stage_number = [0, 23, 29, 29, 31, 32, 33]

    run_names = ["pretrained", "b2diffu_try2", "new_incremental_4_8_12_16_only_lambda_2_fk_4particles", "incremental_branch_lambda_2_fk_4particles"]
    stage_number = [0, 23, 35, 32]

    run_names = ["pretrained", "vanilla_ddpo", "b2diffu_try2", "new_100inc_b2diffu", "5_only_lambda_2_fk_4particles", "all_norm_inc", "only_5_branch_lambda_2_fk_4particles", "5_only_lambda_2_fk_4particle_seed_69"]
    stage_number = [0, 21, 23, 26, 51, 33, 26, 40]

    run_names = ["new_incremental_4_8_12_16_only_lambda_2_fk_4particles", "incremental_branch_lambda_2_fk_4particles", "new_100inc_b2diffu", "all_norm_inc", "incremental_5_10_15_only_lambda_2_fk_4particles", "only_10_branch_lambda_2_fk_4particles","only_10_steps",
"5_only_lambda_2_fk_4particle_seed_69",
"another_only_5_steps",
"norm_all_no_branching_no_selection_only_5_steps",
"b2diffu_try2",
"vanilla_ddpo"]
    stage_number = [35, 32, 26, 33, 41, 31, 13, 40, 34, 14, 23, 21]



    runs = {
        # "pretrained": {"stage": 0, "label": "Pretrained (SD 1.4)"},
        # "vanilla_ddpo": {"stage": 21, "label": "Vanilla DDPO (DDPO | Only 20)"},
        # "new_incremental_4_8_12_16_only_lambda_2_fk_4particles": {"stage": 35, "label": "FK | Incremental"},
        # "incremental_branch_lambda_2_fk_4particles": {"stage": 37, "label": "Branch + FK | Incremental"},
        # "new_100inc_b2diffu": {"stage": 26, "label": "Branch | Incremental"},
        # "all_norm_inc": {"stage": 33, "label": "DDPO | Incremental"},
        # "only_10_branch_lambda_2_fk_4particles": {"stage": 31, "label": "Branch + FK | Only 10"},
        # "only_10_steps": {"stage": 13, "label": "Branch | Only 10"},
        # "b2diffu_try2": {"stage": 23, "label": "B2diff"},
        # "incremental_4_8_12_16_only_lambda_2_fk_4particles": {"stage": 28, "label": "FK | Only 10"},
        # "last_only_10_all_norm": {"stage": 11, "label": "All Norm | Only 10"},
        # "inference_time_fk_steering": {"stage": 0, "label": "inference_time_fk_steering"},
        # "last_only_10_all_norm": {"stage": 13, "label": "All Norm | Only last 10"},
        # "all_norm_only_10": {"stage": 27, "label": "DDPO | Only 10 Steps"}

        # Trash
        # "new_fk_only":  {"stage": 36, "label": "FK | Incremental new"}

        # Template2
        "template2_ddpo": {"stage": 54, "label": "Template 2 DDPO"},
        "template2_b2": {"stage": 42, "label": "Template 2 B2"},
        "template2_branch_fk": {"stage": 69, "label": "Template 2 Branch FK"},
        "template2_fk_only": {"stage": 64, "label": "Template 2 FK only"},
        "template2_branch": {"stage": 53, "label": "Template 2 Branch"},
    }
    run_names = list(runs.keys())
    stage_number = [runs[i]["stage"] for i in runs]
    labels = [runs[i]["label"] for i in runs]
    label_map = {run_name: label for run_name, label in zip(run_names, labels)}

    json_paths = []
    inception_scores = {}
    mean_rewards = {}

    for i in range(len(run_names)):
        json_paths.append(f"/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/{run_names[i]}/stage{stage_number[i]}/clip_rewards.json")
        inception_score_path = f"/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/{run_names[i]}/stage{stage_number[i]}/inception_score.txt"        
        inception_score = extract_mean_from_file(inception_score_path)
        inception_scores[run_names[i]] = inception_score
        clip_file_path = f"/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/{run_names[i]}/stage{stage_number[i]}/clip_rewards.json"        
        mean_rewards[run_names[i]] = extract_mean_reward_from_file(clip_file_path)
    
    # Inception scores (extracted from comments in original code)
    # inception_scores = {
    #     'b2diffu_rl': 1.2858,
    #     'only_5_steps': 1.2845,
    #     'another_only_5_steps': 1.2851,
    #     'no_branch_yes_selection_only_5_steps': 1.2876,
    #     'fk_steering': 1.2701,
    # }
    
    # Line plot with sorted prompts
    plot_clip_rewards(json_paths, save_path="clip_rewards_comparison.png", labels=label_map)
    
    # Heatmap with sorted prompts
    # plot_clip_rewards_heatmap(json_paths, save_path="clip_rewards_heatmap.png", labels=label_map)
    
    # Inception scores visualization
    plot_inception_scores(inception_scores, save_path="inception_scores_comparison.png", labels=label_map)

    plot_mean_rewards(mean_rewards, save_path="mean_rewards_comparison.png", labels=label_map)

    plot_rarity_score(json_paths, save_path="rarity_score_comparison.png", labels=label_map)


In [ ]:
import os
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline

def extract_mean_from_file(file_path):
    if not os.path.exists(file_path): return None
    with open(file_path, 'r') as file:
        content = file.read()
    match = re.search(r'Mean:\s*([\d.]+)', content)
    return float(match.group(1)) if match else None

def plot_reward_vs_inception_monotonic(run_names, base_dir="outputs", smooth=True):
    plt.style.use('seaborn-v0_8-paper') 
    fig, ax = plt.subplots(figsize=(10, 6), dpi=120)
    
    colors = ['#4361ee', '#f72585', '#4cc9f0', '#7209b7', '#3f37c9', '#4895ef']
    plotted_any = False

    for idx, run_name in enumerate(run_names):
        run_dir = os.path.join(base_dir, run_name)
        if not os.path.exists(run_dir): continue
        
        stages, rewards, inceptions = [], [], []
        folders = [f for f in os.listdir(run_dir) if re.match(r"^stage(\d+)$", f)]
        
        for folder in folders:
            stage_num = int(re.search(r"\d+", folder).group())
            path = os.path.join(run_dir, folder)
            r_val = None
            for r_file in ["clip_rewards.json", "geometric_rewards.json"]:
                p = os.path.join(path, r_file)
                if os.path.exists(p):
                    with open(p, 'r') as f:
                        data = json.load(f)
                        r_val = data.get("clip_reward_mean") or data.get("geometric_reward_mean")
            
            is_val = extract_mean_from_file(os.path.join(path, "inception_score.txt"))
            if r_val is not None and is_val is not None:
                stages.append(stage_num); rewards.append(r_val); inceptions.append(is_val)
        
        if not stages: continue
        
        # Sort and Filter for Monotonic Increase in X (Reward)
        sorted_indices = np.argsort(stages)
        x_raw = np.array(rewards)[sorted_indices]
        y_raw = np.array(inceptions)[sorted_indices]
        
        filtered_x, filtered_y = [], []
        max_x_so_far = -float('inf')
        
        for cur_x, cur_y in zip(x_raw, y_raw):
            if cur_x > max_x_so_far: # Only keep points that improve horizontal alignment
                filtered_x.append(cur_x); filtered_y.append(cur_y)
                max_x_so_far = cur_x
        
        if len(filtered_x) < 2: continue
        plotted_any = True
        x, y, color = np.array(filtered_x), np.array(filtered_y), colors[idx % len(colors)]
        
        if smooth and len(x) > 3:
            t = np.linspace(0, 1, len(x))
            t_smooth = np.linspace(0, 1, 300)
            x_sm = make_interp_spline(t, x, k=3)(t_smooth)
            y_sm = make_interp_spline(t, y, k=3)(t_smooth)
            
            ax.plot(x_sm, y_sm, color=color, alpha=0.15, linewidth=6) # Halo
            ax.plot(x_sm, y_sm, color=color, alpha=0.8, linewidth=2.5, label=run_name)
            
            # Directional Arrows
            for i in range(0, len(x_sm)-1, 100):
                ax.annotate('', xy=(x_sm[i+1], y_sm[i+1]), xytext=(x_sm[i], y_sm[i]),
                            arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.6))
        else:
            ax.plot(x, y, color=color, alpha=0.8, linewidth=2, label=run_name, marker='o', markersize=4)

        ax.scatter(x, y, color=color, s=30, edgecolors='white', zorder=5)

    ax.set_title("Monotonic Evaluation Frontier", fontsize=15, pad=20, fontweight='bold')
    ax.set_xlabel("Mean Reward (Alignment) →", fontsize=12)
    ax.set_ylabel("Inception Score (Quality)", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(frameon=True, fontsize=10)
    plt.tight_layout()
    plt.show()

# Usage
# plot_reward_vs_inception_monotonic(["template2_branch_fk", "template2_b2", "template2_ddpo"])


# Run it
plot_reward_vs_inception_monotonic(["b2diffu_try2", "vanilla_ddpo", "incremental_branch_lambda_2_fk_4particles"])

In [ ]:

import os
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


def plot_reward_vs_rarity(run_names, base_dir="outputs",
                          pretrained_run="pretrained", pretrained_stage=0):
    """
    Plot mean CLIP reward vs rarity score for every stage of each run.

    Rarity score (%) = mean(stage_clip_score - pretrained_clip_score) × 100
                       over the 10 lowest-scoring prompts in pretrained.

    Parameters
    ----------
    run_names : list[str]
        Experiment folder names under base_dir.
    base_dir : str
        Root folder that holds per-run sub-directories (default: "outputs").
    pretrained_run : str
        Sub-folder name for the pretrained baseline (default: "pretrained").
    pretrained_stage : int
        Stage number to use for pretrained scores (default: 0).
    """
    if isinstance(run_names, str):
        run_names = [run_names]

    # ── 1. Load pretrained reference scores ──────────────────────────────────
    pretrained_path = os.path.join(
        base_dir, pretrained_run, f"stage{pretrained_stage}", "clip_rewards.json"
    )
    if not os.path.exists(pretrained_path):
        print(f"❌ Pretrained clip_rewards.json not found at: {pretrained_path}")
        return

    with open(pretrained_path, "r") as f:
        pretrained_data = json.load(f)

    pt_prompt_scores = defaultdict(list)
    for prompt, score in zip(pretrained_data["prompts"], pretrained_data["all_scores"]):
        pt_prompt_scores[prompt].append(score)
    pretrained_means = {p: np.mean(s) for p, s in pt_prompt_scores.items()}

    # Identify the 10 lowest-scoring prompts in pretrained (= "rare" prompts)
    prompts_sorted = sorted(
        pretrained_means.keys(),
        key=lambda p: (pretrained_means[p], p),
        reverse=True,
    )
    last_10_prompts = prompts_sorted[-5:]

    # ── 2. Iterate over runs / stages ─────────────────────────────────────────
    markers = ["o", "s", "^", "D", "v", "<", ">", "p", "*"]
    colors  = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
               "#9467bd", "#8c564b", "#e377c2", "#17becf", "#bcbd22"]

    fig, ax = plt.subplots(figsize=(12, 8))
    plotted_any = False

    for idx, run_name in enumerate(run_names):
        run_dir = os.path.join(base_dir, run_name)
        if not os.path.exists(run_dir):
            print(f"⚠️  Run directory not found: {run_dir}")
            continue

        stages, mean_rewards, rarity_scores = [], [], []

        for stage_folder in sorted(os.listdir(run_dir)):
            m = re.match(r"^stage(\d+)$", stage_folder)
            if not m:
                continue
            stage_num  = int(m.group(1))
            stage_path = os.path.join(run_dir, stage_folder)

            clip_path = os.path.join(stage_path, "clip_rewards.json")
            if not os.path.exists(clip_path):
                continue

            with open(clip_path, "r") as f:
                data = json.load(f)

            reward_val = data.get("clip_reward_mean")
            if reward_val is None:
                continue

            # Per-stage prompt means
            stage_prompt_scores = defaultdict(list)
            for prompt, score in zip(data["prompts"], data["all_scores"]):
                stage_prompt_scores[prompt].append(score)
            stage_means = {p: np.mean(s) for p, s in stage_prompt_scores.items()}

            # Rarity = mean delta over rare prompts × 100
            diffs = [
                stage_means.get(p, np.nan) - pretrained_means.get(p, np.nan)
                for p in last_10_prompts
            ]
            rarity = float(np.nanmean(diffs) * 100)

            stages.append(stage_num)
            mean_rewards.append(reward_val)
            rarity_scores.append(rarity)

        if not stages:
            print(f"⚠️  No valid stages found for '{run_name}'.")
            continue

        plotted_any = True

        # Sort chronologically before plotting
        sorted_triplets = sorted(zip(stages, mean_rewards, rarity_scores))
        stages, mean_rewards, rarity_scores = zip(*sorted_triplets)

        color  = colors[idx % len(colors)]
        marker = markers[idx % len(markers)]

        ax.plot(
            mean_rewards, rarity_scores,
            marker=marker, linestyle="-", linewidth=2,
            markersize=8, color=color, label=run_name, alpha=0.8,
        )

        # Annotate each point with its stage number
        for stage, x, y in zip(stages, mean_rewards, rarity_scores):
            y_offset = 12 if idx % 2 == 0 else -18
            ax.annotate(
                f"S{stage}",
                (x, y),
                textcoords="offset points",
                xytext=(0, y_offset),
                ha="center",
                fontsize=9,
                color=color,
                fontweight="bold",
            )

    if not plotted_any:
        print("❌ Nothing to plot.")
        plt.close()
        return

    title = "Mean Reward vs Rarity Score"
    if len(run_names) == 1:
        title += f" ({run_names[0]})"

    ax.axhline(0, color="gray", linewidth=1.2, linestyle="--", alpha=0.6, zorder=0)
    ax.set_title(title, fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("Mean CLIP Reward", fontsize=14)
    ax.set_ylabel("Rarity Score (%)\n(mean CLIP Δ vs pretrained, rare prompts)", fontsize=13)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="best", fontsize=11, framealpha=0.9, shadow=True)

    plt.tight_layout()
    plt.show()


# --- Example Usage ---
runs_to_plot = [
    "b2diffu_try2",
    "vanilla_ddpo",
    "incremental_branch_lambda_2_fk_4particles",
]

plot_reward_vs_rarity(runs_to_plot)


In [ ]:
import os
import re
import json
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def extract_mean_from_file(file_path):
    if not os.path.exists(file_path):
        return None
        
    with open(file_path, 'r') as file:
        content = file.read()
    
    match = re.search(r'Mean:\s*([\d.]+)', content)
    if match:
        return float(match.group(1))
    return None

def plot_reward_vs_inception_3d(run_names, base_dir="outputs"):
    if isinstance(run_names, str):
        run_names = [run_names]
        
    # Setup 3D plot
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    plotted_any = False
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
    
    for idx, run_name in enumerate(run_names):
        run_dir = os.path.join(base_dir, run_name)
        
        stages = []
        mean_rewards = []
        inception_scores = []
        
        if not os.path.exists(run_dir):
            print(f"⚠️ Run directory not found: {run_dir}")
            continue
            
        for stage_folder in os.listdir(run_dir):
            match = re.match(r"^stage(\d+)$", stage_folder)
            if not match:
                continue
                
            stage_num = int(match.group(1))
            stage_path = os.path.join(run_dir, stage_folder)
            
            # 1. Read Reward
            reward_val = None
            clip_path = os.path.join(stage_path, "clip_rewards.json")
            geo_path = os.path.join(stage_path, "geometric_rewards.json")
            
            if os.path.exists(clip_path):
                with open(clip_path, 'r') as f:
                    reward_val = json.load(f).get("clip_reward_mean")
            elif os.path.exists(geo_path):
                with open(geo_path, 'r') as f:
                    reward_val = json.load(f).get("geometric_reward_mean")
            
            # 2. Read Inception Score
            is_path = os.path.join(stage_path, "inception_score.txt")
            is_val = extract_mean_from_file(is_path)
                            
            if reward_val is not None and is_val is not None:
                stages.append(stage_num)
                mean_rewards.append(reward_val)
                inception_scores.append(is_val)
                
        if not stages:
            print(f"⚠️ No valid data for {run_name}.")
            continue
            
        plotted_any = True
            
        # Sort chronologically by stage
        sorted_pairs = sorted(zip(stages, mean_rewards, inception_scores))
        stages, mean_rewards, inception_scores = zip(*sorted_pairs)
        
        color = colors[idx % len(colors)]
        
        # Plot 3D line and scatter points
        ax.plot(stages, mean_rewards, inception_scores, 
                label=run_name, color=color, linewidth=2.5, marker='o', markersize=6, alpha=0.9)

    if not plotted_any:
        print("❌ Nothing to plot.")
        plt.close()
        return

    # Adjust view angle for best visibility
    ax.view_init(elev=20, azim=-45)

    # Set Labels
    ax.set_xlabel('Training Stage', fontsize=12, labelpad=10)
    ax.set_ylabel('Mean Reward', fontsize=12, labelpad=10)
    ax.set_zlabel('Inception Score', fontsize=12, labelpad=10)
    
    title = f"3D Trajectory: Reward vs Inception across Stages"
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Optional styling
    ax.grid(True, linestyle='--', alpha=0.6)
    
    # Legend
    ax.legend(loc='upper left', fontsize=11, framealpha=0.9, shadow=True, bbox_to_anchor=(0.05, 0.95))
    
    plt.tight_layout()
    plt.show()

# --- Example Usage ---
runs_to_plot = [
    "b2diffu_try2",
    "vanilla_ddpo", 
    "incremental_branch_lambda_2_fk_4particles"
]
plot_reward_vs_inception_3d(runs_to_plot)

In [ ]:
# Standalone cell (uses the same pretrained json path)
import json, numpy as np
from collections import defaultdict
from pathlib import Path

pretrained_path = f"/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/pretrained/stage0/clip_rewards.json"

with open(pretrained_path, 'r') as f:
    data = json.load(f)

prompt_scores = defaultdict(list)
for prompt, score in zip(data['prompts'], data['all_scores']):
    prompt_scores[prompt].append(score)

prompt_means = {p: np.mean(s) for p, s in prompt_scores.items()}

# Sort descending → last 10 are the lowest-scoring prompts
prompts_sorted = sorted(prompt_means.keys(), key=lambda p: (prompt_means[p], p), reverse=True)
last_10 = prompts_sorted[-10:]
first_10 = prompts_sorted[:10]

print(first_10)

print("=== Last 10 prompts (lowest CLIP reward in pretrained) ===")
for i, p in enumerate(first_10, 1):
    print(f"  {i:2d}. [{prompt_means[p]:.4f}]  {p}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualize images per run in a grid  (columns = runs, rows = prompts)
# ─────────────────────────────────────────────────────────────────────────────
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image

BASE_DIR = Path("/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs")

def visualize_run_images(
    runs: dict,           # {run_name: {"stage": int, "label": str}}
    filter_prompts=None,
    images_per_prompt: int = 1,
    img_size: float = 2.5,
    title: str = "Generated Images by Run",
    save_path: str = None,
):
    run_names = list(runs.keys())

    # ── 1. Load prompt lists ──────────────────────────────────────────────────
    run_prompt_lists = {}
    for name in run_names:
        stage = runs[name]["stage"]
        pfile = BASE_DIR / name / f"stage{stage}" / "prompts.json"
        with open(pfile) as f:
            run_prompt_lists[name] = json.load(f)

    # ── 2. Determine prompts ──────────────────────────────────────────────────
    if filter_prompts is None:
        seen, unique_prompts = set(), []
        for p in run_prompt_lists[run_names[0]]:
            if p not in seen:
                seen.add(p)
                unique_prompts.append(p)
        display_prompts = unique_prompts
    else:
        display_prompts = list(filter_prompts)

    n_prompts = len(display_prompts)
    n_runs    = len(run_names)
    n_rows    = n_prompts * images_per_prompt

    print(f"Grid: {n_prompts} prompts × {n_runs} runs  ({images_per_prompt} img/cell)")

    # ── 3. Build index ────────────────────────────────────────────────────────
    def build_index(prompt_list):
        from collections import defaultdict
        idx = defaultdict(list)
        for i, p in enumerate(prompt_list):
            idx[p].append(i)
        return idx

    run_indices = {name: build_index(run_prompt_lists[name]) for name in run_names}

    # ── 4. Plot ───────────────────────────────────────────────────────────────
    fig_h = img_size * n_rows + 0.6
    fig_w = img_size * n_runs + 2.5
    fig = plt.figure(figsize=(fig_w, fig_h))

    gs = gridspec.GridSpec(
        n_rows, n_runs, figure=fig,
        hspace=0.05, wspace=0.03,
        left=0.18, right=0.99, top=0.95, bottom=0.01,
    )

    # Column headers — use label instead of run_name
    for col, name in enumerate(run_names):
        ax = fig.add_subplot(gs[0, col])
        ax.set_title(runs[name]["label"], fontsize=9, fontweight="bold", pad=4)
        ax.axis("off")

    for row_prompt, prompt in enumerate(display_prompts):
        for img_k in range(images_per_prompt):
            grid_row = row_prompt * images_per_prompt + img_k

            if img_k == 0:
                short = prompt if len(prompt) <= 35 else prompt[:33] + "…"
                fig.text(
                    0.01, 1 - (grid_row + 0.5) / n_rows * (1 - 0.05) - 0.01,
                    short, va="center", ha="left",
                    fontsize=7, color="#333", transform=fig.transFigure,
                )

            for col, name in enumerate(run_names):
                ax = fig.add_subplot(gs[grid_row, col])
                ax.axis("off")

                stage    = runs[name]["stage"]
                img_dir  = BASE_DIR / name / f"stage{stage}"
                indices  = run_indices[name].get(prompt, [])

                if img_k < len(indices):
                    img_path = img_dir / f"{indices[img_k]:05d}.png"
                    if img_path.exists():
                        ax.imshow(np.array(Image.open(img_path).convert("RGB")))
                    else:
                        ax.text(0.5, 0.5, "missing", ha="center", va="center",
                                fontsize=7, color="red", transform=ax.transAxes)
                        ax.set_facecolor("#f0f0f0")
                else:
                    ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                            fontsize=7, color="#aaa", transform=ax.transAxes)
                    ax.set_facecolor("#fafafa")

    fig.suptitle(title, fontsize=13, fontweight="bold", y=0.98)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()

# ── Example calls ─────────────────────────────────────────────────────────────

rare_prompts = [
   "a shark riding a bike",
   "a dog washing dishes",
   "a butterfly riding a bike",
   "a snake riding a bike",
   "a lizard riding a bike",
   "a spider washing dishes",
   "a beetle riding a bike",
   "a ant riding a bike",
   "a dolphin riding a bike","a whale riding a bike"
]

easy_prompts = ['a kangaroo playing chess', 'a gorilla playing chess', 'a hedgehog playing chess', 'a llama playing chess', 'a fox riding a bike', 'a bear washing dishes', 'a camel playing chess', 'a lion washing dishes', 'a raccoon riding a bike', 'a wolf riding a bike']

all_prompts = ['a kangaroo playing chess', 'a gorilla playing chess', 'a hedgehog playing chess', 'a llama playing chess', 'a fox riding a bike', 'a bear washing dishes', 'a camel playing chess', 'a lion washing dishes', 'a raccoon riding a bike', 'a wolf riding a bike', 'a rat riding a bike', 'a mouse riding a bike', 'a deer washing dishes', 'a bat playing chess', 'a rabbit washing dishes', 'a sheep washing dishes', 'a pig playing chess', 'a chicken playing chess', 'a squirrel riding a bike', 'a bee playing chess', 'a monkey washing dishes', 'a tiger washing dishes', 'a frog playing chess', 'a goose playing chess', 'a turtle playing chess', 'a horse washing dishes', 'a cat washing dishes', 'a bird washing dishes', 'a fish riding a bike', 'a cow washing dishes', 'a turkey playing chess', 'a zebra washing dishes', 'a fly playing chess', 'a duck playing chess', 'a goat washing dishes', 'a shark riding a bike', 'a dog washing dishes', 'a butterfly riding a bike', 'a snake riding a bike', 'a lizard riding a bike', 'a spider washing dishes', 'a beetle riding a bike', 'a ant riding a bike', 'a dolphin riding a bike', 'a whale riding a bike']


custom_prompts = ['brown_banana']


visualize_run_images(
    # runs = {
    #     # "pretrained": {"stage": 0, "label": "Pretrained (SD 1.4)"},
    #     "vanilla_ddpo": {"stage": 21, "label": "Vanilla DDPO (DDPO | Only 20)"},
    #     "b2diffu_try2": {"stage": 23, "label": "B2diff"},
    #     "incremental_branch_lambda_2_fk_4particles": {"stage": 37, "label": "Branch + FK | Incremental"},
    #     "new_incremental_4_8_12_16_only_lambda_2_fk_4particles": {"stage": 35, "label": "FK | Incremental"},
    #     # "new_100inc_b2diffu": {"stage": 26, "label": "Branch | Incremental"},
    #     # "all_norm_inc": {"stage": 33, "label": "DDPO | Incremental"},
    #     # "only_10_branch_lambda_2_fk_4particles": {"stage": 31, "label": "Branch + FK | Only 10"},
    #     # "only_10_steps": {"stage": 13, "label": "Branch | Only 10"},
    #     # "incremental_4_8_12_16_only_lambda_2_fk_4particles": {"stage": 28, "label": "FK | Only 10"}
    # },

    runs = {
        # "pretrained": {"stage": 0, "label": "Pretrained (SD 1.4)"},
        # "vanilla_ddpo": {"stage": 21, "label": "Vanilla DDPO (DDPO | Only 20)"},
        # "new_incremental_4_8_12_16_only_lambda_2_fk_4particles": {"stage": 35, "label": "FK | Incremental"},
        # "incremental_branch_lambda_2_fk_4particles": {"stage": 37, "label": "Branch + FK | Incremental"},
        # "new_100inc_b2diffu": {"stage": 26, "label": "Branch | Incremental"},
        # "all_norm_inc": {"stage": 33, "label": "DDPO | Incremental"},
        # "only_10_branch_lambda_2_fk_4particles": {"stage": 31, "label": "Branch + FK | Only 10"},
        # "only_10_steps": {"stage": 13, "label": "Branch | Only 10"},
        # "b2diffu_try2": {"stage": 23, "label": "B2diff"},
        # "incremental_4_8_12_16_only_lambda_2_fk_4particles": {"stage": 28, "label": "FK | Only 10"},
        # "last_only_10_all_norm": {"stage": 11, "label": "All Norm | Only 10"},
        # "inference_time_fk_steering": {"stage": 0, "label": "inference_time_fk_steering"},
        # "last_only_10_all_norm": {"stage": 13, "label": "All Norm | Only last 10"},
        # "all_norm_only_10": {"stage": 27, "label": "DDPO | Only 10 Steps"}

        # Trash
        # "new_fk_only":  {"stage": 36, "label": "FK | Incremental new"}

        # Template2
        "template2_ddpo": {"stage": 54, "label": "Template 2 DDPO"},
        "template2_b2": {"stage": 42, "label": "Template 2 B2"},
        "template2_branch_fk": {"stage": 69, "label": "Template 2 Branch FK"},
        "template2_fk_only": {"stage": 64, "label": "Template 2 FK only"},
        "template2_branch": {"stage": 53, "label": "Template 2 Branch"},
    },
    # filter_prompts=custom_prompts,
    images_per_prompt=10,      # show 2 different images per (prompt, run)
    title="Rare prompts — 2 examples per run",
    save_path="grid_rare_prompts_test.png",
)

In [ ]:
#!/usr/bin/env python3
"""
Analyze vanishing point for a single image.
Detects lines using LSD, fits a vanishing point via least-squares,
draws extended rays and the VP on the image, and saves the result.

Usage:
    python scripts/analyze_vp_single.py --image path/to/image.png
    python scripts/analyze_vp_single.py --image path/to/image.png --show_labels --out output.jpg
    python scripts/analyze_vp_single.py --image path/to/image.png --exclude_ids 0,2 --red 1,3
"""

import os
import cv2
import math
import argparse
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageEnhance


# ---------------------------------------------------------------------------
# Line detection
# ---------------------------------------------------------------------------

def detect_lsd_lines(img_pil, min_length=50):
    """Detect lines using OpenCV's LSD (parameter-free, more precise than Hough)."""
    img_np = np.array(img_pil)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    lsd = cv2.createLineSegmentDetector(0)
    lines, _, _, _ = lsd.detect(gray)
    result = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if math.hypot(x2 - x1, y2 - y1) > min_length:
                result.append((x1, y1, x2, y2))
    return result


def filter_depth_lines(raw_lines):
    """Keep only diagonal lines (exclude near-horizontal and near-vertical)."""
    depth_lines = []
    for line in raw_lines:
        x1, y1, x2, y2 = line
        angle = abs(math.degrees(math.atan2(y2 - y1, x2 - x1)))
        is_horz = angle < 15 or angle > 165
        is_vert = 80 < angle < 100
        if not is_horz and not is_vert:
            depth_lines.append(line)
    depth_lines.sort(key=lambda l: math.hypot(l[2] - l[0], l[3] - l[1]), reverse=True)
    return depth_lines


def merge_lines(lines, theta_thresh=8, rho_thresh=30):
    """Merge nearby parallel line segments into single representatives."""
    merged = []
    used = [False] * len(lines)
    for i in range(len(lines)):
        if used[i]:
            continue
        x1, y1, x2, y2 = lines[i]
        dx, dy = x2 - x1, y2 - y1
        angle = math.degrees(math.atan2(dy, dx)) % 180
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        cluster = [lines[i]]
        used[i] = True
        for j in range(i + 1, len(lines)):
            if used[j]:
                continue
            jx1, jy1, jx2, jy2 = lines[j]
            jcx, jcy = (jx1 + jx2) / 2, (jy1 + jy2) / 2
            jdx, jdy = jx2 - jx1, jy2 - jy1
            jangle = math.degrees(math.atan2(jdy, jdx)) % 180
            angle_diff = abs(angle - jangle)
            if angle_diff > 90:
                angle_diff = 180 - angle_diff
            A, B = -dy, dx
            C = dy * x1 - dx * y1
            perp_dist = abs(A * jcx + B * jcy + C) / (math.hypot(A, B) + 1e-6)
            if angle_diff < theta_thresh and perp_dist < rho_thresh:
                cluster.append(lines[j])
                used[j] = True
        merged.append(max(cluster, key=lambda l: math.hypot(l[2] - l[0], l[3] - l[1])))
    return merged


# ---------------------------------------------------------------------------
# Vanishing point calculation
# ---------------------------------------------------------------------------

def calculate_vp(lines):
    """Least-squares vanishing point + RMS error (pixels)."""
    if len(lines) < 2:
        return 0.0, (0.0, 0.0)
    A, B = [], []
    for x1, y1, x2, y2 in lines:
        dx, dy = x2 - x1, y2 - y1
        norm = math.hypot(dx, dy)
        a, b = dy / norm, -dx / norm
        A.append([a, b])
        B.append(a * x1 + b * y1)
    A, B = np.array(A), np.array(B)
    try:
        vp, _, _, _ = np.linalg.lstsq(A, B, rcond=None)
        vp_x, vp_y = vp
        sq_errs = [(abs(A[i, 0] * vp_x + A[i, 1] * vp_y - B[i]) ** 2) for i in range(len(lines))]
        rms = math.sqrt(sum(sq_errs) / len(sq_errs))
        return rms, (vp_x, vp_y)
    except np.linalg.LinAlgError:
        return 999.0, (0.0, 0.0)


# ---------------------------------------------------------------------------
# Drawing
# ---------------------------------------------------------------------------

def draw_analysis(img_pil, lines_with_ids, vp, show_labels=False, red_indices=None, darken=0.5):
    """Draw VP rays and the vanishing point on the image."""
    if red_indices is None:
        red_indices = []

    enhancer = ImageEnhance.Brightness(img_pil)
    out = enhancer.enhance(darken).copy()
    draw = ImageDraw.Draw(out, "RGBA")

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
    except Exception:
        font = ImageFont.load_default()

    w, h = out.size

    for orig_idx, (x1, y1, x2, y2) in lines_with_ids:
        vx, vy = x2 - x1, y2 - y1
        length = math.hypot(vx, vy)
        if length == 0:
            continue
        vx, vy = vx / length, vy / length
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        # Extend line to a long ray in both directions
        ex1, ey1 = mx - vx * 5000, my - vy * 5000
        ex2, ey2 = mx + vx * 5000, my + vy * 5000

        color = (255, 50, 50, 230) if orig_idx in red_indices else (50, 255, 50, 230)
        draw.line([(ex1, ey1), (ex2, ey2)], fill=color, width=2)

        if show_labels:
            lx, ly = mx + vx * 30, my + vy * 30
            if 0 <= lx < w and 0 <= ly < h:
                for ddx, ddy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    draw.text((lx + ddx, ly + ddy), str(orig_idx), fill="black", font=font)
                label_color = "red" if orig_idx in red_indices else "yellow"
                draw.text((lx, ly), str(orig_idx), fill=label_color, font=font)

    # Draw VP dot if inside image
    vpx, vpy = vp
    if 0 <= vpx < w and 0 <= vpy < h:
        r = 8
        draw.ellipse([vpx - r, vpy - r, vpx + r, vpy + r],
                     fill="green", outline="white", width=2)

    return out


# ---------------------------------------------------------------------------
# Score overlay
# ---------------------------------------------------------------------------

def draw_score_overlay(img_pil, rms_error, vp):
    """Draw a semi-transparent bar at the bottom with the RMS score."""
    draw = ImageDraw.Draw(img_pil, "RGBA")
    w, h = img_pil.size
    bar_h = 60
    draw.rectangle([0, h - bar_h, w, h], fill=(0, 0, 0, 180))

    try:
        font_lg = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 26)
        font_sm = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 18)
    except Exception:
        font_lg = font_sm = ImageFont.load_default()

    draw.text((15, h - bar_h + 8), "VP Analysis", fill=(100, 255, 100), font=font_lg)
    vp_str = f"VP: ({vp[0]:.1f}, {vp[1]:.1f})   RMS Error: {rms_error:.2f} px"
    draw.text((15, h - bar_h + 36), vp_str, fill=(200, 200, 200), font=font_sm)
    return img_pil

import os
from PIL import Image

# ── Inputs ──────────────────────────────────────────────────────────────────
img_name="00000"
image_path   = f"/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/pretrained_baseline_clip/{img_name}.png"
out_path     = f"/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/pretrained_baseline_clip/{img_name}_vp.png"
show_labels  = True
exclude_ids  = []        # e.g. [0, 2, 5]
red_ids      = []        # e.g. [1, 3]
max_lines    = 12
min_length   = 25  # pixels
darken       = 1.0     # background brightness (0–1)
# ────────────────────────────────────────────────────────────────────────────

print(f"Loading image: {image_path}")
img = Image.open(image_path).convert("RGB")

# Detect → filter → merge
raw    = detect_lsd_lines(img, min_length=min_length)
depth  = filter_depth_lines(raw)
merged = merge_lines(depth)[:max_lines]

# Apply exclusions, keep original IDs
lines_with_ids = [(idx, line) for idx, line in enumerate(merged) if idx not in exclude_ids]
valid_lines    = [line for _, line in lines_with_ids]

print(f"  {len(raw)} raw → {len(depth)} depth → {len(merged)} merged → {len(valid_lines)} used")

# Compute VP
rms, vp = calculate_vp(valid_lines)
print(f"  VP: ({vp[0]:.1f}, {vp[1]:.1f})   RMS Error: {rms:.2f} px")

# Draw
result = draw_analysis(img, lines_with_ids, vp,
                       show_labels=show_labels,
                       red_indices=red_ids,
                       darken=darken)
result = draw_score_overlay(result, rms, vp)

# Save & display
os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
result.save(out_path, quality=95)
print(f"  Saved to: {out_path}")

display(result)   # renders inline in Jupyter


In [ ]:
# python scripts/inference/eval_pretrained_reward.py \
#   --output_dir temp_outputs/pretrained_geometric \
#   --prompt_file configs/prompt/template4_train.json \
#   --reward_fn geometric \
#   --num_images 24 \
#   --batch_size 4

In [ ]:
img_name="00005"

In [ ]:
import os
import math
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageEnhance
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


# ─── VP helper functions (same as single-image analysis) ─────────────────────

def detect_lsd_lines(img_pil, min_length=50):
    img_np = np.array(img_pil)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    lsd = cv2.createLineSegmentDetector(0)
    lines, _, _, _ = lsd.detect(gray)
    result = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if math.hypot(x2 - x1, y2 - y1) > min_length:
                result.append((x1, y1, x2, y2))
    return result


def filter_depth_lines(raw_lines):
    depth_lines = []
    for line in raw_lines:
        x1, y1, x2, y2 = line
        angle = abs(math.degrees(math.atan2(y2 - y1, x2 - x1)))
        is_horz = angle < 15 or angle > 165
        is_vert = 80 < angle < 100
        if not is_horz and not is_vert:
            depth_lines.append(line)
    depth_lines.sort(key=lambda l: math.hypot(l[2] - l[0], l[3] - l[1]), reverse=True)
    return depth_lines


def merge_lines(lines, theta_thresh=8, rho_thresh=30):
    merged = []
    used = [False] * len(lines)
    for i in range(len(lines)):
        if used[i]:
            continue
        x1, y1, x2, y2 = lines[i]
        dx, dy = x2 - x1, y2 - y1
        angle = math.degrees(math.atan2(dy, dx)) % 180
        cluster = [lines[i]]
        used[i] = True
        for j in range(i + 1, len(lines)):
            if used[j]:
                continue
            jx1, jy1, jx2, jy2 = lines[j]
            jcx, jcy = (jx1 + jx2) / 2, (jy1 + jy2) / 2
            jdx, jdy = jx2 - jx1, jy2 - jy1
            jangle = math.degrees(math.atan2(jdy, jdx)) % 180
            angle_diff = abs(angle - jangle)
            if angle_diff > 90:
                angle_diff = 180 - angle_diff
            A, B = -dy, dx
            C = dy * x1 - dx * y1
            perp_dist = abs(A * jcx + B * jcy + C) / (math.hypot(A, B) + 1e-6)
            if angle_diff < theta_thresh and perp_dist < rho_thresh:
                cluster.append(lines[j])
                used[j] = True
        merged.append(max(cluster, key=lambda l: math.hypot(l[2] - l[0], l[3] - l[1])))
    return merged


def calculate_vp(lines):
    if len(lines) < 2:
        return 0.0, (0.0, 0.0)
    A, B = [], []
    for x1, y1, x2, y2 in lines:
        dx, dy = x2 - x1, y2 - y1
        norm = math.hypot(dx, dy)
        a, b = dy / norm, -dx / norm
        A.append([a, b])
        B.append(a * x1 + b * y1)
    A, B = np.array(A), np.array(B)
    try:
        vp, _, _, _ = np.linalg.lstsq(A, B, rcond=None)
        vp_x, vp_y = vp
        sq_errs = [(abs(A[i, 0] * vp_x + A[i, 1] * vp_y - B[i]) ** 2) for i in range(len(lines))]
        rms = math.sqrt(sum(sq_errs) / len(sq_errs))
        return rms, (vp_x, vp_y)
    except np.linalg.LinAlgError:
        return 999.0, (0.0, 0.0)


def draw_analysis(img_pil, lines_with_ids, vp, show_labels=False, darken=1.0):
    enhancer = ImageEnhance.Brightness(img_pil)
    out = enhancer.enhance(darken).copy()
    draw = ImageDraw.Draw(out, "RGBA")
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
    except Exception:
        font = ImageFont.load_default()
    w, h = out.size
    for orig_idx, (x1, y1, x2, y2) in lines_with_ids:
        vx, vy = x2 - x1, y2 - y1
        length = math.hypot(vx, vy)
        if length == 0:
            continue
        vx, vy = vx / length, vy / length
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ex1, ey1 = mx - vx * 5000, my - vy * 5000
        ex2, ey2 = mx + vx * 5000, my + vy * 5000
        draw.line([(ex1, ey1), (ex2, ey2)], fill=(50, 255, 50, 230), width=2)
        if show_labels:
            lx, ly = mx + vx * 30, my + vy * 30
            if 0 <= lx < w and 0 <= ly < h:
                draw.text((lx, ly), str(orig_idx), fill="yellow", font=font)
    vpx, vpy = vp
    if 0 <= vpx < w and 0 <= vpy < h:
        r = 8
        draw.ellipse([vpx - r, vpy - r, vpx + r, vpy + r],
                     fill="green", outline="white", width=2)
    return out


def analyze_image(image_path, min_length=25, max_lines=12):
    """Run full VP analysis on one image; return (annotated PIL image, rms, vp)."""
    img = Image.open(image_path).convert("RGB")
    raw    = detect_lsd_lines(img, min_length=min_length)
    depth  = filter_depth_lines(raw)
    merged = merge_lines(depth)[:max_lines]
    lines_with_ids = [(idx, line) for idx, line in enumerate(merged)]
    valid_lines    = [line for _, line in lines_with_ids]
    rms, vp = calculate_vp(valid_lines)
    annotated = draw_analysis(img, lines_with_ids, vp)
    return annotated, rms, vp


# ─── Grid visualisation function ─────────────────────────────────────────────

def visualize_grid(
    image_paths,
    ncols=4,
    cell_size=(3.5, 3.5),      # inches per cell
    analyze_vp=True,           # overlay VP lines
    show_labels=False,
    min_length=25,
    max_lines=12,
    title=None,
    title_fontsize=14,
):
    """
    Display a grid of images, optionally with vanishing-point overlays.

    Parameters
    ----------
    image_paths : list[str]
        Absolute paths to PNG/JPG images.
    ncols : int
        Number of columns in the grid.
    cell_size : (float, float)
        Width and height of each grid cell in inches.
    analyze_vp : bool
        If True, run VP analysis and draw lines/dot on each image.
    show_labels : bool
        If True, draw line-index labels (only relevant when analyze_vp=True).
    min_length : int
        Minimum pixel length for LSD-detected lines.
    max_lines : int
        Maximum number of lines to use per image.
    title : str or None
        Optional super-title for the whole figure.
    title_fontsize : int
        Font size for the super-title.

    Returns
    -------
    fig : matplotlib Figure
    """
    n = len(image_paths)
    if n == 0:
        raise ValueError("image_paths is empty")

    nrows = math.ceil(n / ncols)
    fig_w = ncols * cell_size[0]
    fig_h = nrows * cell_size[1] + (0.6 if title else 0)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(fig_w, fig_h),
                             squeeze=False)

    for idx, ax in enumerate(axes.flat):
        ax.axis("off")
        if idx >= n:
            continue
        path = image_paths[idx]
        img = Image.open(path).convert("RGB")

        if analyze_vp:
            raw    = detect_lsd_lines(img, min_length=min_length)
            depth  = filter_depth_lines(raw)
            merged = merge_lines(depth)[:max_lines]
            lines_with_ids = [(i, l) for i, l in enumerate(merged)]
            valid_lines    = [l for _, l in lines_with_ids]
            rms, vp = calculate_vp(valid_lines)
            display_img = draw_analysis(img, lines_with_ids, vp,
                                        show_labels=show_labels)
            subtitle = f"RMS={rms:.2f}"
        else:
            display_img = img
            subtitle = os.path.basename(path)

        ax.imshow(display_img)
        ax.set_title(subtitle, fontsize=8, pad=2)

    if title:
        fig.suptitle(title, fontsize=title_fontsize, y=1.01)

    plt.tight_layout(pad=0.4)
    plt.show()
    return fig


# ─── Example usage ───────────────────────────────────────────────────────────
# Collect images from a directory (change path as needed)
img_dir = "outputs/pretrained_baseline_clip"

image_paths = sorted(
    [os.path.join(img_dir, f)
     for f in os.listdir(img_dir)
     if f.endswith(".png") and not f.endswith("_vp.png")]
)[:16]   # show first 16 images

fig = visualize_grid(
    image_paths,
    ncols=4,
    cell_size=(3.5, 3.5),
    analyze_vp=True,
    title="Pretrained SD — VP Analysis (first 16 images)",
)
